<a href="https://colab.research.google.com/github/mhtjsh/ViT-ImageSegmentation-Training/blob/Primary/q1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ENV

In [1]:
import os
import random
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset, random_split
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

### Reproducibility

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    # for CUDA
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # deterministic options (may slow down)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, " Device:", device)
if torch.cuda.is_available():
    try:
        print("CUDA device name:", torch.cuda.get_device_name(0))
    except Exception:
        pass

PyTorch: 2.8.0+cu126  Device: cuda
CUDA device name: Tesla T4


In [4]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

In [5]:
train_transform = T.Compose([
    T.RandomCrop(32, padding=4),    # common for CIFAR: random crop with 4px padding
    T.RandomHorizontalFlip(p=0.5),  # horizontal flip
    T.ToTensor(),                   # convert PIL -> tensor [0,1]
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

### Creating Validation, Training and Test Dataset with dataloaders

In [6]:
# Colab cell 3 — dataloader helper
from torchvision.datasets import CIFAR10

def get_cifar10_dataloaders(data_root="./data",
                            batch_size=128,
                            val_size=5000,
                            num_workers=4,
                            seed=42,
                            download=True):

    train_aug_ds = CIFAR10(root=data_root, train=True, download=download, transform=train_transform)
    train_noaug_ds = CIFAR10(root=data_root, train=True, download=False, transform=val_transform)
    test_ds = CIFAR10(root=data_root, train=False, download=False, transform=val_transform)

    num_train = len(train_aug_ds)
    assert num_train == 50000,

    rng = np.random.RandomState(seed)
    indices = rng.permutation(num_train)
    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

    train_subset = Subset(train_aug_ds, train_indices)
    val_subset = Subset(train_noaug_ds, val_indices)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)

    classes = train_aug_ds.classes
    dataset_sizes = {"train": len(train_subset), "val": len(val_subset), "test": len(test_ds)}
    return train_loader, val_loader, test_loader, classes, dataset_sizes

# checks
train_loader, val_loader, test_loader, classes, sizes = get_cifar10_dataloaders(batch_size=128, val_size=5000)
print("Dataset sizes:", sizes)
print("Classes:", classes)


100%|██████████| 170M/170M [00:03<00:00, 43.1MB/s]


Dataset sizes: {'train': 45000, 'val': 5000, 'test': 10000}
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


### Patch Embeddings

In [8]:
import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=128):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.embed_dim = embed_dim
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))

    def forward(self, x):
        B = x.shape[0]  # batch size
        x = self.proj(x)  # (B, embed_dim, H/p, W/p)
        x = x.flatten(2)  # (B, embed_dim, num_patches)
        x = x.transpose(1, 2)  # (B, num_patches, embed_dim)
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        x = torch.cat((cls_tokens, x), dim=1)  # (B, 1+num_patches, embed_dim)
        x = x + self.pos_embed
        return x


In [9]:
# test patch embedding
patch_size = 4
embed_dim = 128

patch_embed = PatchEmbedding(img_size=32, patch_size=patch_size,
                             in_channels=3, embed_dim=embed_dim)
images, labels = next(iter(train_loader))
images = images.to(device)

patch_embed = patch_embed.to(device)
out = patch_embed(images)

print("Input batch shape:", images.shape)
print("Output shape:", out.shape)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Input batch shape: torch.Size([128, 3, 32, 32])
Output shape: torch.Size([128, 65, 128])


### Multi Head Self Attention Layer

In [11]:
# Colab cell — Multi-Head Self Attention
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=128, num_heads=8, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        assert embed_dim % num_heads == 0,

        self.head_dim = embed_dim // num_heads

        # Linear layers for Q, K, V
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.out = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        out = attn @ v
        out = out.transpose(1, 2).reshape(B, N, D)
        out = self.out(out)
        out = self.proj_drop(out)
        return out


In [15]:
import torch
import torch.nn as nn

# Dummy batch
batch_size, seq_len, d_model = 128, 65, 128
x = torch.randn(batch_size, seq_len, d_model)  # original input

# Simulate MHSA output (same shape)
mhsa_out = torch.randn(batch_size, seq_len, d_model)

# Residual + LayerNorm
layernorm1 = nn.LayerNorm(d_model)
x = layernorm1(x + mhsa_out)
print("After Residual + LayerNorm:", x.shape)


After Residual + LayerNorm: torch.Size([128, 65, 128])


### Wrapped One Encoder Block

In [16]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim=128, num_heads=8, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.mlp_ratio = mlp_ratio

        # Multi-Head Self Attention
        self.mhsa = MultiHeadSelfAttention(embed_dim, num_heads, dropout=dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        # Feed Forward Network (MLP)
        hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # --- Multi-Head Self Attention + Residual + Norm
        x = self.norm1(x + self.dropout(self.mhsa(x)))

        # --- Feed Forward Network + Residual + Norm
        x = self.norm2(x + self.dropout(self.mlp(x)))

        return x

In [17]:
# Dummy batch (same as before)
batch_size, seq_len, d_model = 128, 65, 128
x = torch.randn(batch_size, seq_len, d_model).to(device)

# Create one encoder block
encoder_block = TransformerEncoderBlock(embed_dim=d_model, num_heads=8).to(device)

# Forward pass
out = encoder_block(x)
print("Output shape after 1 encoder block:", out.shape)


Output shape after 1 encoder block: torch.Size([128, 65, 128])


### Stacking the N Encoder Blocks

In [18]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 num_classes=10, embed_dim=128, depth=6, num_heads=8,
                 mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.encoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)  # (B, num_patches+1, embed_dim)
        for block in self.encoder_blocks:
            x = block(x)
        cls_token = x[:, 0]  # shape: (B, embed_dim)
        cls_token = self.norm(cls_token)
        logits = self.head(cls_token)
        return logits


In [19]:
# ViT hyperparameters
img_size = 32
patch_size = 4
embed_dim = 128
depth = 6
num_heads = 8
num_classes = 10
dropout = 0.1

# Instantiate model
model = VisionTransformer(img_size, patch_size, in_channels=3,
                          num_classes=num_classes, embed_dim=embed_dim,
                          depth=depth, num_heads=num_heads, dropout=dropout).to(device)

# Check summary
dummy_input = torch.randn(2, 3, img_size, img_size).to(device)
out = model(dummy_input)
print("Output logits shape:", out.shape)


Output logits shape: torch.Size([2, 10])


### Loss & Optimisation

In [20]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)


In [21]:
model = model.to(device)
criterion = criterion.to(device)


In [22]:
def accuracy(output, target):
    preds = output.argmax(dim=1)
    return (preds == target).float().mean()


In [23]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_acc = 0.0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Metrics
        running_loss += loss.item() * images.size(0)
        running_acc += accuracy(outputs, labels) * images.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc.item()


In [24]:
@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_acc = 0.0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        running_acc += accuracy(outputs, labels) * images.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc.item()


## Baseline Test

In [25]:
# Training settings
num_epochs = 20
best_val_acc = 0.0

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Scheduler
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_vit_model.pth")

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


Epoch [1/20] | Train Loss: 1.8230, Train Acc: 0.3188 | Val Loss: 1.6326, Val Acc: 0.4132
Epoch [2/20] | Train Loss: 1.4622, Train Acc: 0.4649 | Val Loss: 1.3026, Val Acc: 0.5284
Epoch [3/20] | Train Loss: 1.2935, Train Acc: 0.5310 | Val Loss: 1.2009, Val Acc: 0.5648
Epoch [4/20] | Train Loss: 1.1963, Train Acc: 0.5648 | Val Loss: 1.0886, Val Acc: 0.6090
Epoch [5/20] | Train Loss: 1.1252, Train Acc: 0.5918 | Val Loss: 1.0525, Val Acc: 0.6184
Epoch [6/20] | Train Loss: 1.0677, Train Acc: 0.6158 | Val Loss: 0.9852, Val Acc: 0.6442
Epoch [7/20] | Train Loss: 1.0261, Train Acc: 0.6308 | Val Loss: 0.9828, Val Acc: 0.6472
Epoch [8/20] | Train Loss: 0.9844, Train Acc: 0.6474 | Val Loss: 0.9123, Val Acc: 0.6596
Epoch [9/20] | Train Loss: 0.9476, Train Acc: 0.6602 | Val Loss: 0.9010, Val Acc: 0.6826
Epoch [10/20] | Train Loss: 0.9105, Train Acc: 0.6736 | Val Loss: 0.8399, Val Acc: 0.7048
Epoch [11/20] | Train Loss: 0.8869, Train Acc: 0.6823 | Val Loss: 0.8118, Val Acc: 0.7062
Epoch [12/20] | Tra

**76% Improvement**

## Improvements
### 1. Data Augmentation

- Definifng data augmentation

In [26]:
import torchvision.transforms as transforms

# Training transforms with augmentation
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

# Validation / Test transforms (no augmentation)
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])


In [27]:
import torchvision
from torch.utils.data import DataLoader, random_split

# Full CIFAR-10 training set
full_train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)

# Split into training + validation (45000 / 5000)
train_set, val_set = random_split(full_train_set, [45000, 5000])

# Test set
test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=val_transform)

# DataLoaders
batch_size = 128
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)


In [28]:
import torch.optim as optim

# Reset optimizer
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

In [29]:
num_epochs = 20
best_val_acc = 0.0

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Scheduler step
    scheduler.step()

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_vit_model_augmented.pth")

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


Epoch [1/20] | Train Loss: 0.7222, Train Acc: 0.7439 | Val Loss: 0.7224, Val Acc: 0.7470
Epoch [2/20] | Train Loss: 0.7012, Train Acc: 0.7496 | Val Loss: 0.6967, Val Acc: 0.7612
Epoch [3/20] | Train Loss: 0.6961, Train Acc: 0.7540 | Val Loss: 0.7149, Val Acc: 0.7498
Epoch [4/20] | Train Loss: 0.6727, Train Acc: 0.7608 | Val Loss: 0.6506, Val Acc: 0.7722
Epoch [5/20] | Train Loss: 0.6635, Train Acc: 0.7636 | Val Loss: 0.6587, Val Acc: 0.7712
Epoch [6/20] | Train Loss: 0.6526, Train Acc: 0.7699 | Val Loss: 0.6738, Val Acc: 0.7644
Epoch [7/20] | Train Loss: 0.6388, Train Acc: 0.7734 | Val Loss: 0.6735, Val Acc: 0.7698
Epoch [8/20] | Train Loss: 0.6285, Train Acc: 0.7756 | Val Loss: 0.6502, Val Acc: 0.7728
Epoch [9/20] | Train Loss: 0.6172, Train Acc: 0.7784 | Val Loss: 0.6683, Val Acc: 0.7666
Epoch [10/20] | Train Loss: 0.5981, Train Acc: 0.7876 | Val Loss: 0.6485, Val Acc: 0.7762
Epoch [11/20] | Train Loss: 0.5892, Train Acc: 0.7910 | Val Loss: 0.6438, Val Acc: 0.7748
Epoch [12/20] | Tra

**81% Improvement**

### 2. Depth/Width Trade-offs
- Defining depth/width configs

In [32]:
configs = [
    {"depth": 4, "embed_dim": 128},
    {"depth": 6, "embed_dim": 128},  # baseline
    {"depth": 8, "embed_dim": 128},
    {"depth": 6, "embed_dim": 192},
    {"depth": 6, "embed_dim": 256},
    {"depth": 8, "embed_dim": 192},
]

results = {}


In [35]:
for cfg in configs:
    depth = cfg["depth"]
    embed_dim = cfg["embed_dim"]
    print(f"\n--- Training ViT with depth={depth}, embed_dim={embed_dim} ---")
    best_val = train_vit_config(depth, embed_dim, num_epochs=20)
    results[f"depth{depth}_width{embed_dim}"] = best_val

print("\n--- Depth/Width Trade-off Results ---")
for k, v in results.items():
    print(f"{k}: Best Val Acc = {v:.4f}")



--- Training ViT with depth=4, embed_dim=128 ---
Config depth=4, embed_dim=128 | Epoch [1/20] | Train Acc: 0.3064 | Val Acc: 0.4126
Config depth=4, embed_dim=128 | Epoch [2/20] | Train Acc: 0.4495 | Val Acc: 0.5038
Config depth=4, embed_dim=128 | Epoch [3/20] | Train Acc: 0.5187 | Val Acc: 0.5588
Config depth=4, embed_dim=128 | Epoch [4/20] | Train Acc: 0.5545 | Val Acc: 0.5840
Config depth=4, embed_dim=128 | Epoch [5/20] | Train Acc: 0.5772 | Val Acc: 0.5860
Config depth=4, embed_dim=128 | Epoch [6/20] | Train Acc: 0.5961 | Val Acc: 0.6002
Config depth=4, embed_dim=128 | Epoch [7/20] | Train Acc: 0.6123 | Val Acc: 0.6182
Config depth=4, embed_dim=128 | Epoch [8/20] | Train Acc: 0.6249 | Val Acc: 0.6354
Config depth=4, embed_dim=128 | Epoch [9/20] | Train Acc: 0.6388 | Val Acc: 0.6464
Config depth=4, embed_dim=128 | Epoch [10/20] | Train Acc: 0.6494 | Val Acc: 0.6492
Config depth=4, embed_dim=128 | Epoch [11/20] | Train Acc: 0.6585 | Val Acc: 0.6778
Config depth=4, embed_dim=128 | Epo